In [ ]:
#| default_exp read

# read

> The queries. What happened in a session, and what has ever happened to a file.

`Ledger` reads the folder and answers questions about it. It holds every record in memory,
because a year of heavy use is single-digit megabytes, and rereads only when a shard has changed
on disk.

Two questions carry the package. `sessions` is the log: what ran, on what, and how it went.
`trail` is the blame: every session that has ever touched one file, newest first. The second is
the one that is hard to answer today.

In [ ]:
#| export
import time
from pathlib import Path

from fastcore.basics import AttrDict, patch
from fastcore.foundation import L

from panjika.core import DETAIL, LEDGER, Home, fold, records

## Loading

The cache key is the name, size and modification time of every shard. A hook appending in
another process changes all three, so a long-lived reader in an editor sees new records without
being told to look.

In [ ]:
#| export
def shard_stamp(home, tier=LEDGER):
    "What changes when a shard does. The cache key."
    out = []
    for p in home.shards(tier):
        try: st = p.stat()
        except OSError: continue
        out.append((p.name, st.st_size, st.st_mtime_ns))
    return tuple(out)


class Ledger:
    "One ledger, read. Rereads a tier when a shard on disk has changed."

    def __init__(self, home=None, start='.'):
        self.home = home if isinstance(home, Home) else Home(home, start)
        self._cache, self._stamps = {}, {}

    def __repr__(self): return f'Ledger({self.home.path}, {len(self.all())} records)'

    @property
    def exists(self): return self.home.exists

    def all(self, tier=LEDGER):
        "Every record in a tier, oldest first."
        stamp = shard_stamp(self.home, tier)
        if self._stamps.get(tier) != stamp:
            self._cache[tier], self._stamps[tier] = records(self.home, tier), stamp
        return self._cache[tier]

    def of(self, kind, tier=LEDGER):
        "Every record of one kind."
        return self.all(tier).filter(lambda r: r.kind == kind)

## Sessions

In [ ]:
#| export
#: Fields whose first value is the one that describes the session, not their last.
OPENING = ('prompt',)


def asks(recs, sid):
    "Every prompt put to one session, in order, each with where it came from."
    return [(r.prompt, r.get('origin') or 'human') for r in recs
            if r.get('session') == sid and r.get('prompt')]


def headline(asked):
    """What a session was for: the first thing a person asked it.

    A harness injects prompts of its own -- a notification, a reminder, a resumed turn -- and
    one of those is regularly the first record in a transcript. Opening on one of them
    describes the session by something nobody asked for.
    """
    return next((p for p, o in asked if o != 'injected'), asked[0][0] if asked else '')


def counters(steps):
    """What a session's steps add up to: how many worked, how many did not, and which tools.

    The count is `n_steps` rather than `steps`, because `steps` on a session row is the list of
    the step records themselves and a count would be standing where the list belongs.
    """
    ok = sum(1 for s in steps if s.get('ok', True))
    tools, actions = {}, {}
    for s in steps:
        tools[s.get('tool', '?')] = tools.get(s.get('tool', '?'), 0) + 1
        actions[s.get('action', 'other')] = actions.get(s.get('action', 'other'), 0) + 1
    return {'n_steps': len(steps), 'steps_ok': ok, 'steps_fail': len(steps) - ok,
            'secs': round(sum(float(s.get('secs') or 0) for s in steps), 2),
            'tools': dict(sorted(tools.items(), key=lambda kv: -kv[1])),
            'actions': dict(sorted(actions.items(), key=lambda kv: -kv[1]))}


def _since(value):
    "Seconds since the epoch for `7d`, `12h`, `30m`, a count of days, or a timestamp."
    if value in (None, ''): return 0
    if isinstance(value, (int, float)) and value > 10_000_000: return float(value)
    text = str(value).strip().lower()
    units = {'d': 86400, 'h': 3600, 'm': 60, 'w': 604800, 's': 1}
    n, unit = (text[:-1], text[-1]) if text[-1:] in units else (text, 'd')
    try: return time.time() - float(n) * units[unit]
    except ValueError: return 0

In [ ]:
#| export
@patch
def sessions(self:Ledger, limit=20, harness='', since='', repo='', path='', status=''):
    "Session rows, newest first. Every filter is optional and they compose."
    recs = self.of('session')
    rows = fold(recs, first=OPENING)
    for r in rows:
        if (asked := asks(recs, r.session)): r.prompt = headline(asked)
    if path: rows = rows.filter(lambda r: r.session in self.sessions_touching(path))
    for field, want in (('harness', harness), ('repo', repo), ('status', status)):
        if want: rows = rows.filter(lambda r, f=field, w=want: r.get(f) == w)
    if since:
        after = _since(since)
        rows = rows.filter(lambda r: (r.get('started') or r.get('at') or 0) >= after)
    rows = rows.sorted(key=lambda r: r.get('started') or r.get('at') or 0, reverse=True)
    return rows if not limit else rows[:int(limit)]


@patch
def sessions_touching(self:Ledger, path):
    "The ids of every session that touched `path`."
    p = str(path)
    return {r.session for r in self.of('touch') if r.get('path') == p}

In [ ]:
#| export
@patch
def files(self:Ledger, session):
    """What one session did to each file it touched, one row per path.

    A session that edits a file five times leaves five touches of the same net change against
    `HEAD`, so the last one is the whole of what it did and the rest are its working.
    """
    out = {}
    for r in self.of('touch'):
        if r.session == session: out[r.path] = r
    return L(out.values()).sorted(key=lambda r: r.path)


@patch
def session(self:Ledger, sid):
    """One session, with everything recorded against it.

    The counters are computed here rather than written. A hook fires once per tool call in its
    own process and cannot keep a running total, so counting is the reader's job.
    """
    recs = self.of('session')
    rows = fold(recs, first=OPENING).filter(lambda r: r.session == sid)
    row = AttrDict(rows[0]) if rows else AttrDict(kind='session', session=sid, id=sid)
    asked = asks(recs, sid)
    row['prompts'] = L(asked)
    row['prompt'] = headline(asked) if asked else row.get('prompt', '')
    row['last_prompt'] = asked[-1][0] if asked else ''
    for kind in ('step', 'touch', 'commit', 'note'):
        row[f'{kind}s'] = self.of(kind).filter(lambda r: r.session == sid).sorted(
            key=lambda r: r.get('at') or 0)
    row['files'] = self.files(sid)
    row.update(counters(row.steps))
    row['seconds'] = round((row.get('ended') or row.get('at') or 0) - (row.get('started') or 0), 1)
    return row

## The trail over a file

The question this package exists for. Every session that ever touched one path, newest first,
each with what it did to that file and the commits recorded against it.

In [ ]:
#| export
@patch
def trail(self:Ledger, path, limit=50):
    "Every session that touched `path`, newest first, with its touch of that file."
    p, out = str(path), []
    by_session = {}
    for r in self.of('touch'):
        if r.get('path') == p: by_session[r.session] = r      # the last touch is the net one
    rows = {r.session: r for r in fold(self.of('session'))}
    commits = {}
    for c in self.of('commit'):
        if p in (c.get('files') or ()): commits.setdefault(c.session, []).append(c)
    for sid, touch in by_session.items():
        row = AttrDict(rows.get(sid) or {'kind': 'session', 'session': sid, 'id': sid})
        out.append(AttrDict(row, touch=touch, commits=L(commits.get(sid) or []),
                            at=touch.get('at') or row.get('at') or 0))
    out = L(out).sorted(key=lambda r: r.at, reverse=True)
    return out if not limit else out[:int(limit)]


@patch
def touched(self:Ledger, since='', limit=200):
    "Every path the ledger knows about, most recently touched first."
    after, seen = _since(since), {}
    for r in self.of('touch'):
        if (r.get('at') or 0) < after: continue
        seen[r.path] = max(seen.get(r.path, 0), r.get('at') or 0)
    return L(sorted(seen.items(), key=lambda kv: -kv[1])[:int(limit)])

In [ ]:
#| export
@patch
def search(self:Ledger, query, limit=20):
    "Sessions whose prompt, title or touched paths mention `query`."
    q = str(query).lower()
    hits = {r.session for r in self.of('touch') if q in str(r.get('path', '')).lower()}
    out = []
    for row in fold(self.of('session')):
        blob = ' '.join(str(row.get(k, '')) for k in ('prompt', 'title', 'model', 'harness'))
        if q in blob.lower() or row.session in hits: out.append(row)
    return L(out).sorted(key=lambda r: r.get('started') or 0, reverse=True)[:int(limit)]


@patch
def detail(self:Ledger, record_id):
    "The machine-local half of one record: whole arguments, whole output, exact changed lines."
    for r in self.all(DETAIL):
        if r.id == record_id: return r
    return None


@patch
def stats(self:Ledger):
    "What this ledger holds."
    kinds = {}
    for r in self.all(): kinds[r.kind] = kinds.get(r.kind, 0) + 1
    rows = fold(self.of('session'))
    return AttrDict(path=str(self.home.path), records=len(self.all()), sessions=len(rows),
                    harnesses=sorted({r.get('harness', '') for r in rows} - {''}),
                    files=len({r.path for r in self.of('touch')}),
                    detail=len(self.all(DETAIL)), **kinds)

## Trying it

In [ ]:
import subprocess, tempfile
from fastcore.test import test_eq
from panjika.write import Scribe

def _git(root, *a): subprocess.run(['git', *a], cwd=root, capture_output=True, check=True)

d = Path(tempfile.mkdtemp())/'proj'; d.mkdir(parents=True)
_git(d, 'init', '-q', '-b', 'main')
_git(d, 'config', 'user.email', 'a@b.c'); _git(d, 'config', 'user.name', 'T')
(d/'app.py').write_text('def add(a, b):\n    return a + b\n')
(d/'util.py').write_text('X = 1\n')
_git(d, 'add', '-A'); _git(d, 'commit', '-qm', 'first')

def a_session(harness, prompt, path, text):
    sc = Scribe(home=d/'.panjika', start=d)
    sc.home.init()
    sc.begin(harness, model='opus-5', prompt=prompt)
    (d/path).write_text(text)
    sc.touch(d/path, 'edit', sc.step('Edit', target=path, secs=0.2))
    sc.end('done', turns=1)
    return sc.session

one = a_session('claude-code', 'handle strings', 'app.py',
                'def add(a, b):\n    if isinstance(a, str): return a + str(b)\n    return a + b\n')
two = a_session('codex', 'add a docstring', 'app.py',
                'def add(a, b):\n    "Add two things."\n    if isinstance(a, str): return a + str(b)\n    return a + b\n')
three = a_session('ramabana', 'bump the constant', 'util.py', 'X = 2\n')

led = Ledger(d/'.panjika')
test_eq(len(led.sessions()), 3)
test_eq([r.harness for r in led.sessions()], ['ramabana', 'codex', 'claude-code'])

In [ ]:
# three harnesses, one ledger, and the trail over one file knows about two of them
rows = led.trail('app.py')
test_eq([r.harness for r in rows], ['codex', 'claude-code'])
test_eq([r.prompt for r in rows], ['add a docstring', 'handle strings'])
test_eq(rows[0].touch.path, 'app.py')

In [ ]:
test_eq([p for p, _ in led.touched()], ['util.py', 'app.py'])
test_eq(len(led.sessions(harness='codex')), 1)
test_eq(len(led.sessions(path='util.py')), 1)
test_eq([r.harness for r in led.search('util')], ['ramabana'])
test_eq(led.session(one).files[0].path, 'app.py')
test_eq(led.stats().files, 2)

# the counters are computed from the records, not written by a hook that cannot count
row = led.session(one)
test_eq((row.n_steps, row.steps_ok, row.steps_fail), (1, 1, 0))
test_eq(row.tools, {'Edit': 1})
test_eq([s.tool for s in row.steps], ['Edit'])   # the list, not the count

In [ ]:
# A session is described by one record per prompt and folding lets the last value win, so a
# long session would otherwise be headlined by whatever was said to it most recently. What it
# was *for* is the first thing a person asked, and a harness injects prompts of its own.
sc = Scribe(home=d/'.panjika', start=d)
sc.begin('claude-code', model='opus-5')
sc.write('session', prompt='<task-notification>a subagent finished', origin='injected')
sc.write('session', prompt='port the parser to the new API', origin='human')
sc.write('session', prompt='now fix the lint', origin='human')
sc.end('done')

row = Ledger(d/'.panjika').session(sc.session)
test_eq(row.prompt, 'port the parser to the new API')   # not the injected one, not the last one
test_eq(row.last_prompt, 'now fix the lint')            # the later asks are kept, in order
test_eq(len(row.prompts), 3)
test_eq((row.status, row.model), ('done', 'opus-5'))    # only the ask is pinned to its first value
test_eq(Ledger(d/'.panjika').sessions()[0].prompt, 'port the parser to the new API')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()